<img src="../../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Lab: Exploring Loan Rejections Data - Solution
---

In this lab we'll look at data about **loan applications** from a company called Lending Club that facilitated peer-to-peer loans.  Here is some text from their website a few years ago, to give you a sense of their purpose:

_"Since 2007, we’ve been bringing borrowers and investors together, transforming the way people access credit. Over the last 10 years, we've helped millions of people take control of their debt, grow their small businesses, and invest for the future."_

#### 1. Load the `pandas` library

In [ ]:
import pandas as pd

#### 2. Read in the dataset and take a look at it

In [ ]:
df = pd.read_csv("../data/rejections.csv.gz")
df.head()

#### 3. How many rows are there?

In [ ]:
len(df)

#### 4. What are each column's data types?

In [ ]:
df.dtypes

#### 5. Investigate which columns have missing values

In [ ]:
df.isnull().sum()

#### 6. What are the smallest and largest loan amounts?

In [ ]:
print(df["Amount Requested"].min(), df["Amount Requested"].max())

#### 7. Find the rows where the requested amount is equal to the minimum and maximum values to investigate these further. Is the amount a valid value in those cases?

Decide what to do with those rows - are they reasonable data? If you don't think so, drop the relevant rows.

In [ ]:
df[df["Amount Requested"] == 0]

In [ ]:
df[df["Amount Requested"] == df["Amount Requested"].max()]

Conclusion: the 0 loan amounts look like bad data, but it's possible someone tried asking for a 1.4 million loan for  a car...

In [ ]:
df = df[df["Amount Requested"] > 0]

#### 8. What are the different values in "employment length" and how many applications are there in each category?

In [ ]:
df["Employment Length"].value_counts().sort_index()

We can use `sort_index()` to order by the category, not the count. Not perfect because they're strings, so 10 is between 1 and 2, but we get the gist!

#### 9. You've been asked to estimate what % of loan applications are related to debt consolidation

- drop rows that have no loan title
- convert each string in the "loan title" column to be fully uppercase
- count how many rows have the word "DEBT" in the loan title
- work out what this is as a % of your total dataset

In [ ]:
df = df.dropna(subset=["Loan Title"])

df["loan_title_upper"] = df["Loan Title"].str.upper()

debt_count = len(df[df["loan_title_upper"].str.contains("DEBT")])

print(debt_count)

print(100*(debt_count / len(df)))

#### 10. BONUS: Expand on the above by trying multiple keywords at once

- identify a set of (uppercase) keywords to search for in the loan titles and store them in a Python list
- as above, make sure your loan titles are uppercase
- search for rows containing any of your keywords

You could do this with the multiple filter syntax:

```python
df[df[(<condition 1>) & (<condition 2>)...]]
```

but that means you can't store your keywords in a list and amend them without amending the above code.

Instead, one approach is to create a new column to track whether a row matches any of the keywords. Then looping through your list of keywords, for each row that matches that particular keyword, set the value of this column to `True`. So the idea is:

- Say you're looking for words "DEBT" and "CONSOLIDATE"
- First, identify all rows that contain the word "DEBT" and update a column, say "matches_keyword", to `True`.
- Then do the same for "CONSOLIDATE". Most of the ones that have this word will already have the "matches_keyword" set to `True`, but the idea is to also catch the ones that don't
- You can do this for any number of keywords
- When you're done, you can use the "matches_keyword" column to tell you how many rows match **at least one** of your keywords

In [ ]:
keywords = ["DEBT", "CONSOLIDATE", "CONSOLIDATING"]

for word in keywords:
    df.loc[df["loan_title_upper"].str.contains(word), "matches_keyword"] = True

df.fillna({"matches_keyword": False}, inplace=True)
    
total_count = len(df[df["matches_keyword"]])

print(total_count)
print(100*(total_count / len(df)))

Another way to get the % from this columns is using `value_counts`:

In [ ]:
df["matches_keyword"].value_counts() / len(df)